# 03: Enrichment Analysis

This notebook handles:
- Testing if high pLLPS proteins preferentially interact with each other
- Chi-squared enrichment analysis
- High-High vs High-Low interaction patterns
- Statistical significance testing

**Inputs:**
- `results/string_interactions_matched.csv` - From notebook 02
- `results/full_dataset.csv` - From notebook 01

**Outputs:**
- `results/enrichment_results.json` - Statistical test results
- `results/enrichment_matrix.csv` - Interaction matrix (High/Med/Low x High/Med/Low)
- Visualization plots

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import llps_functions as lf
from scipy.stats import chi2_contingency

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Previous Results

In [ ]:
# Load matched interactions
matched_df = lf.load_analysis_result('string_interactions_matched', format='csv')
df_full = lf.load_analysis_result('full_dataset', format='csv')

print(f"\n📊 Loaded data:")
print(f"   Matched interactions: {len(matched_df)}")
print(f"   Full dataset: {len(df_full)} proteins")
print(f"   Complete pairs (both in dataset): {matched_df['both_in_dataset'].sum()}")

## 2. High-High vs High-Low Enrichment Analysis

Test if high pLLPS proteins preferentially interact with other high pLLPS proteins.

In [ ]:
# Set threshold
HIGH_THRESHOLD = 0.7

# Analyze enrichment
print(f"\n🔬 Testing for enrichment of high-high pLLPS interactions...")
print(f"   High pLLPS threshold: {HIGH_THRESHOLD}")

enrichment_results = lf.analyze_interaction_enrichment(
    matched_df, 
    threshold=HIGH_THRESHOLD
)

if enrichment_results:
    print(f"\n📊 Results:")
    print(f"   High-High interactions (observed): {enrichment_results['high_high_observed']}")
    print(f"   High-High interactions (expected): {enrichment_results['high_high_expected']:.1f}")
    print(f"   Enrichment factor: {enrichment_results['enrichment']:.2f}x")
    print(f"   Chi-squared statistic: {enrichment_results['chi2_stat']:.2f}")
    print(f"   P-value: {enrichment_results['p_value']:.4e}")
    
    if enrichment_results['p_value'] < 0.05:
        print(f"\n   ✅ Significant enrichment detected (p < 0.05)")
    else:
        print(f"\n   ⚠️  No significant enrichment (p >= 0.05)")
else:
    print("\n⚠️  Could not perform enrichment analysis")

In [ ]:
# Visualize observed vs expected
if enrichment_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar plot
    categories = ['High-High', 'High-Other']
    observed = [enrichment_results['high_high_observed'], enrichment_results['high_other_observed']]
    expected = [enrichment_results['high_high_expected'], enrichment_results['high_other_expected']]
    
    x = np.arange(len(categories))
    width = 0.35
    
    axes[0].bar(x - width/2, observed, width, label='Observed', alpha=0.8)
    axes[0].bar(x + width/2, expected, width, label='Expected', alpha=0.8)
    axes[0].set_xlabel('Interaction Type')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Observed vs Expected Interactions')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(categories)
    axes[0].legend()
    
    # Enrichment visualization
    axes[1].bar(['Enrichment'], [enrichment_results['enrichment']], color='steelblue', alpha=0.8)
    axes[1].axhline(1.0, color='red', linestyle='--', label='No enrichment')
    axes[1].set_ylabel('Enrichment Factor')
    axes[1].set_title(f'High-High Interaction Enrichment\n(p={enrichment_results["p_value"]:.4e})')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('results/enrichment_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Saved plot: results/enrichment_analysis.png")

## 3. Full Interaction Matrix (High/Medium/Low)

Create a 3x3 matrix of interactions between High/Medium/Low pLLPS proteins.

In [ ]:
# Full matrix analysis
print("\n🔬 Analyzing full interaction matrix (High/Med/Low)...")

matrix_results = lf.analyze_interaction_matrix(
    matched_df, 
    df_full,
    high_threshold=0.7,
    low_threshold=0.4
)

if matrix_results:
    print(f"\n📊 Interaction Matrix Results:")
    print(f"   Chi-squared statistic: {matrix_results['chi2_stat']:.2f}")
    print(f"   P-value: {matrix_results['p_value']:.4e}")
    print(f"   Degrees of freedom: {matrix_results['dof']}")
    
    if matrix_results['p_value'] < 0.05:
        print(f"\n   ✅ Significant deviation from expected (p < 0.05)")
    else:
        print(f"\n   ⚠️  No significant deviation (p >= 0.05)")
else:
    print("\n⚠️  Could not perform matrix analysis")

In [ ]:
# Display matrices
if matrix_results:
    print("\n📋 Observed Interaction Matrix:")
    display(matrix_results['observed_matrix'])
    
    print("\n📋 Expected Interaction Matrix:")
    display(matrix_results['expected_matrix'])
    
    print("\n📋 Enrichment Ratios (Observed/Expected):")
    enrichment_matrix = matrix_results['observed_matrix'] / matrix_results['expected_matrix']
    enrichment_matrix = enrichment_matrix.fillna(0)
    display(enrichment_matrix)

In [ ]:
# Visualize matrices as heatmaps
if matrix_results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Observed
    sns.heatmap(matrix_results['observed_matrix'], annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar_kws={'label': 'Count'})
    axes[0].set_title('Observed Interactions')
    axes[0].set_xlabel('Protein 2 pLLPS Class')
    axes[0].set_ylabel('Protein 1 pLLPS Class')
    
    # Expected
    sns.heatmap(matrix_results['expected_matrix'], annot=True, fmt='.1f', cmap='Oranges', ax=axes[1], cbar_kws={'label': 'Count'})
    axes[1].set_title('Expected Interactions')
    axes[1].set_xlabel('Protein 2 pLLPS Class')
    axes[1].set_ylabel('Protein 1 pLLPS Class')
    
    # Enrichment ratio
    enrichment_matrix = matrix_results['observed_matrix'] / matrix_results['expected_matrix']
    enrichment_matrix = enrichment_matrix.fillna(0)
    sns.heatmap(enrichment_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=1.0, ax=axes[2], cbar_kws={'label': 'Fold Enrichment'})
    axes[2].set_title('Enrichment (Obs/Exp)')
    axes[2].set_xlabel('Protein 2 pLLPS Class')
    axes[2].set_ylabel('Protein 1 pLLPS Class')
    
    plt.tight_layout()
    plt.savefig('results/interaction_matrix_heatmaps.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✅ Saved plot: results/interaction_matrix_heatmaps.png")

## 4. Save Results

In [ ]:
# Save enrichment results
if enrichment_results:
    lf.save_analysis_result(enrichment_results, 'enrichment_results', format='json')

# Save matrix results
if matrix_results:
    # Save observed matrix
    lf.save_analysis_result(matrix_results['observed_matrix'], 'enrichment_matrix_observed', format='csv')
    
    # Save expected matrix
    lf.save_analysis_result(matrix_results['expected_matrix'], 'enrichment_matrix_expected', format='csv')
    
    # Save enrichment ratios
    enrichment_matrix = matrix_results['observed_matrix'] / matrix_results['expected_matrix']
    lf.save_analysis_result(enrichment_matrix.fillna(0), 'enrichment_matrix_ratios', format='csv')

print("\n" + "="*60)
print("✅ All results saved successfully!")
print("="*60)

In [ ]:
# List saved files
lf.list_saved_results()

## Summary

✅ **Completed:**
1. Tested for high-high pLLPS interaction enrichment
2. Performed chi-squared statistical tests
3. Created 3x3 interaction matrices (High/Med/Low)
4. Generated heatmap visualizations
5. Saved all results to `results/` directory

**Next step:** Run `04_network_analysis.ipynb` for network topology and hub analysis.